# IDS GCN-BiGRU (Minimal Notebook with SBOA Hyperparameter Tuning)
Single training pipeline with Bidirectional GRU layers and optional Secretary Bird Optimization Algorithm (SBOA) for hyperparameter tuning.

**Quick Start:**
1. Run all cells in **Section 1** (Setup) - Imports & Config
2. Run all cells in **Section 2** (Data Preprocessing) - Load & Prepare Data  
3. Run cells in **Section 3** (Hyperparameter Tuning) - Optional SBOA optimization
4. Run cells in **Section 4** (Model Training) - Build, train, and evaluate

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, GRU, Bidirectional, Dense, Dropout, BatchNormalization, Reshape
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

## Section 1: Setup (Required)

Import all necessary libraries and define SBOA optimization classes.

In [2]:
# Configuration: Hyperparameter Tuning using Secretary Bird Optimization Algorithm
# Set enable_sboa = True to run hyperparameter optimization (takes 1-2 hours)
# Set enable_sboa = False to use default hyperparameters

ENABLE_SBOA = False  # Change to True to enable hyperparameter tuning

# SBOA Configuration
SBOA_CONFIG = {
    'population_size': 20,           # Number of candidate solutions
    'iterations': 30,                # Optimization iterations (max)
    'early_stop_patience': 10,       # Stop if no improvement for N iterations
    'validation_split': 0.2,         # Use 20% of training data for SBOA validation
}

print('=' * 70)
print('HYPERPARAMETER TUNING CONFIGURATION')
print('=' * 70)
print(f'Enable SBOA Optimization: {ENABLE_SBOA}')
if ENABLE_SBOA:
    print(f'Population Size: {SBOA_CONFIG["population_size"]}')
    print(f'Max Iterations: {SBOA_CONFIG["iterations"]}')
    print(f'Early Stop Patience: {SBOA_CONFIG["early_stop_patience"]} iterations')
print('=' * 70)

HYPERPARAMETER TUNING CONFIGURATION
Enable SBOA Optimization: False


In [3]:
# Secretary Bird Optimization Algorithm (SBOA) Implementation
# Adapted for hyperparameter tuning with early stopping and validation loss tracking

class SecretaryBirdOptimization:
    """
    Secretary Bird Optimization Algorithm for hyperparameter tuning.
    Inspired by secretary bird's behavior in hunting and problem-solving.
    """
    
    def __init__(self, search_space, objective_func, population_size=20, iterations=30, 
                 early_stop_patience=10, verbose=True):
        """
        Initialize SBOA optimizer.
        
        Args:
            search_space: Dict of parameter bounds {param: (min, max)}
            objective_func: Function to minimize (returns validation loss)
            population_size: Number of candidate solutions
            iterations: Maximum iterations
            early_stop_patience: Stop if no improvement for N iterations
            verbose: Print progress
        """
        self.search_space = search_space
        self.objective_func = objective_func
        self.population_size = population_size
        self.iterations = iterations
        self.early_stop_patience = early_stop_patience
        self.verbose = verbose
        self.param_names = list(search_space.keys())
        self.best_solution = None
        self.best_fitness = float('inf')
        self.fitness_history = []
        
    def initialize_population(self):
        """Initialize random population within search space bounds."""
        population = []
        for _ in range(self.population_size):
            solution = {}
            for param, (min_val, max_val) in self.search_space.items():
                solution[param] = np.random.uniform(min_val, max_val)
            population.append(solution)
        return population
    
    def evaluate_fitness(self, solution):
        """Evaluate fitness (validation loss) of a solution."""
        try:
            fitness = self.objective_func(solution)
            return fitness
        except Exception as e:
            if self.verbose:
                print(f"  Error evaluating solution: {e}")
            return float('inf')
    
    def update_position(self, current, best, worst, iteration):
        """
        Update position using Secretary Bird hunting behavior.
        Secretary birds use visual hunting and strategic positioning.
        """
        updated = {}
        w = 0.7 - (iteration / self.iterations) * 0.2  # Inertia weight decay
        c1 = 2.0 - (iteration / self.iterations) * 1.5  # Cognitive component decay
        
        for param in self.param_names:
            min_val, max_val = self.search_space[param]
            
            # Secretary bird behavior: balance exploration and exploitation
            if np.random.rand() < 0.6:
                # Attraction to best solution (exploitation)
                updated[param] = (current[param] + 
                                 c1 * np.random.rand() * (best[param] - current[param]))
            else:
                # Avoidance of worst solution (exploration)
                updated[param] = (current[param] + 
                                 w * np.random.rand() * (current[param] - worst[param]))
            
            # Clip to bounds
            updated[param] = np.clip(updated[param], min_val, max_val)
        
        return updated
    
    def optimize(self):
        """Run SBOA optimization."""
        if self.verbose:
            print('\n' + '=' * 70)
            print('SECRETARY BIRD OPTIMIZATION ALGORITHM')
            print('=' * 70)
        
        # Initialize population
        population = self.initialize_population()
        fitness_values = [self.evaluate_fitness(sol) for sol in population]
        
        best_idx = np.argmin(fitness_values)
        self.best_solution = population[best_idx].copy()
        self.best_fitness = fitness_values[best_idx]
        
        best_no_improve = 0
        
        for iteration in range(self.iterations):
            worst_idx = np.argmax(fitness_values)
            worst_solution = population[worst_idx]
            
            # Update each solution
            new_population = []
            new_fitness = []
            
            for i, solution in enumerate(population):
                updated = self.update_position(solution, self.best_solution, worst_solution, iteration)
                updated_fitness = self.evaluate_fitness(updated)
                
                # Keep better solution
                if updated_fitness < fitness_values[i]:
                    new_population.append(updated)
                    new_fitness.append(updated_fitness)
                else:
                    new_population.append(solution)
                    new_fitness.append(fitness_values[i])
            
            population = new_population
            fitness_values = new_fitness
            
            # Update best solution
            best_idx = np.argmin(fitness_values)
            if fitness_values[best_idx] < self.best_fitness:
                self.best_solution = population[best_idx].copy()
                self.best_fitness = fitness_values[best_idx]
                best_no_improve = 0
            else:
                best_no_improve += 1
            
            self.fitness_history.append(self.best_fitness)
            
            if self.verbose and (iteration + 1) % 5 == 0:
                print(f'Iteration {iteration + 1}/{self.iterations} | Best Loss: {self.best_fitness:.6f}')
            
            # Early stopping
            if best_no_improve >= self.early_stop_patience:
                if self.verbose:
                    print(f'Early stopping at iteration {iteration + 1} (no improvement for {best_no_improve} iterations)')
                break
        
        if self.verbose:
            print(f'\n✓ Optimization complete!')
            print(f'Best Loss: {self.best_fitness:.6f}')
            print(f'Best Hyperparameters:')
            for param, value in self.best_solution.items():
                print(f'  {param}: {value:.6f}')
            print('=' * 70)
        
        return self.best_solution, self.best_fitness

In [4]:
# Hyperparameter Search Spaces for GCN and BiGRU

# Default hyperparameters (used when SBOA is disabled)
DEFAULT_HYPERPARAMS = {
    # BiGRU Architecture
    'bigru_units_1': 32,          # First BiGRU layer units
    'bigru_units_2': 64,          # Second BiGRU layer units
    'bigru_units_3': 64,          # Third BiGRU layer units
    'dense_units_1': 128,         # First dense layer units
    'dense_units_2': 64,          # Second dense layer units
    'dense_units_3': 32,          # Third dense layer units
    'dropout_rate_1': 0.3,        # Dropout after first dense
    'dropout_rate_2': 0.2,        # Dropout after second dense
    'dropout_rate_3': 0.2,        # Dropout after third dense
    'learning_rate': 1e-3,        # Adam optimizer learning rate
    'batch_size': 256,            # Training batch size
}

# SBOA Search Space for hyperparameter optimization
SBOA_SEARCH_SPACE = {
    # BiGRU layer units (16-128 range)
    'bigru_units_1': (16, 128),
    'bigru_units_2': (32, 256),
    'bigru_units_3': (32, 256),
    
    # Dense layer units (64-512 range)
    'dense_units_1': (64, 512),
    'dense_units_2': (32, 256),
    'dense_units_3': (16, 128),
    
    # Dropout rates (0.1-0.5 range)
    'dropout_rate_1': (0.1, 0.5),
    'dropout_rate_2': (0.1, 0.5),
    'dropout_rate_3': (0.1, 0.5),
    
    # Learning rate (1e-5 to 1e-2)
    'learning_rate': (1e-5, 1e-2),
    
    # Batch size (32-512 range, log scale)
    'batch_size': (32, 512),
}

print('Hyperparameter Search Spaces Defined:')
print(f'  Total parameters to optimize: {len(SBOA_SEARCH_SPACE)}')

Hyperparameter Search Spaces Defined:
  Total parameters to optimize: 11


In [5]:
# Objective Function for SBOA: Build and evaluate model with given hyperparameters
def build_bigru_model(hyperparams, n_features, n_classes):
    """Build BiGRU model with specified hyperparameters."""
    model = Sequential([
        Input(shape=(n_features,)),
        Reshape((n_features, 1)),
        Reshape((1, n_features)),
        
        Bidirectional(GRU(int(hyperparams['bigru_units_1']), return_sequences=True)),
        BatchNormalization(),
        
        Bidirectional(GRU(int(hyperparams['bigru_units_2']), return_sequences=True)),
        BatchNormalization(),
        
        Bidirectional(GRU(int(hyperparams['bigru_units_3']))),
        BatchNormalization(),
        
        Dense(int(hyperparams['dense_units_1']), activation='relu'),
        Dropout(hyperparams['dropout_rate_1']),
        
        Dense(int(hyperparams['dense_units_2']), activation='relu'),
        Dropout(hyperparams['dropout_rate_2']),
        
        Dense(int(hyperparams['dense_units_3']), activation='relu'),
        Dropout(hyperparams['dropout_rate_3']),
        
        Dense(n_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=hyperparams['learning_rate']),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model


def objective_function_sboa(hyperparams, x_train_sboa, y_train_sboa, x_val_sboa, y_val_sboa):
    """
    Objective function for SBOA: Train model and return validation loss.
    Lower loss is better (minimization problem).
    """
    try:
        # Build model with hyperparameters
        n_features = x_train_sboa.shape[1]
        n_classes = y_train_sboa.shape[1]
        model = build_bigru_model(hyperparams, n_features, n_classes)
        
        # Train with minimal epochs for SBOA evaluation (fast feedback)
        callbacks_sboa = [
            EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        ]
        
        history = model.fit(
            x_train_sboa, y_train_sboa,
            validation_data=(x_val_sboa, y_val_sboa),
            epochs=25,  # Reduced epochs for fast evaluation
            batch_size=int(hyperparams['batch_size']),
            callbacks=callbacks_sboa,
            verbose=0
        )
        
        # Return validation loss as fitness (to minimize)
        val_loss = min(history.history['val_loss'])
        return val_loss
        
    except Exception as e:
        print(f"Error in objective function: {e}")
        return float('inf')  # Return worst fitness on error


print('✓ Model building and objective function defined')

✓ Model building and objective function defined


In [6]:
# Wrapper function: Get optimized or default hyperparameters
def get_hyperparameters(enable_sboa, x_train, y_train, sboa_config, verbose=True):
    """
    Get hyperparameters either by SBOA optimization or default values.
    
    Args:
        enable_sboa: Boolean to enable Secretary Bird Optimization
        x_train: Training features
        y_train: Training labels
        sboa_config: SBOA configuration dict
        verbose: Print progress
    
    Returns:
        hyperparams: Dictionary of optimized hyperparameters
    """
    if not enable_sboa:
        if verbose:
            print('\n' + '=' * 70)
            print('Using Default Hyperparameters (SBOA Disabled)')
            print('=' * 70)
            for param, value in DEFAULT_HYPERPARAMS.items():
                print(f'  {param}: {value}')
            print('=' * 70)
        return DEFAULT_HYPERPARAMS.copy()
    
    # SBOA Optimization
    if verbose:
        print('\n' + '=' * 70)
        print('Running Secretary Bird Optimization for Hyperparameter Tuning')
        print('=' * 70)
    
    # Split training data for SBOA validation
    x_sboa_train, x_sboa_val, y_sboa_train, y_sboa_val = train_test_split(
        x_train, y_train,
        test_size=sboa_config['validation_split'],
        random_state=42,
        stratify=np.argmax(y_train, axis=1)
    )
    
    # Create objective function with specific data
    def sboa_objective(hyperparams):
        return objective_function_sboa(hyperparams, x_sboa_train, y_sboa_train, 
                                       x_sboa_val, y_sboa_val)
    
    # Initialize and run SBOA
    optimizer = SecretaryBirdOptimization(
        search_space=SBOA_SEARCH_SPACE,
        objective_func=sboa_objective,
        population_size=sboa_config['population_size'],
        iterations=sboa_config['iterations'],
        early_stop_patience=sboa_config['early_stop_patience'],
        verbose=verbose
    )
    
    best_hyperparams, best_loss = optimizer.optimize()
    
    # Convert to appropriate types
    optimized = {
        'bigru_units_1': int(best_hyperparams['bigru_units_1']),
        'bigru_units_2': int(best_hyperparams['bigru_units_2']),
        'bigru_units_3': int(best_hyperparams['bigru_units_3']),
        'dense_units_1': int(best_hyperparams['dense_units_1']),
        'dense_units_2': int(best_hyperparams['dense_units_2']),
        'dense_units_3': int(best_hyperparams['dense_units_3']),
        'dropout_rate_1': float(best_hyperparams['dropout_rate_1']),
        'dropout_rate_2': float(best_hyperparams['dropout_rate_2']),
        'dropout_rate_3': float(best_hyperparams['dropout_rate_3']),
        'learning_rate': float(best_hyperparams['learning_rate']),
        'batch_size': int(best_hyperparams['batch_size']),
    }
    
    return optimized

---

## Section 3: Hyperparameter Tuning (Optional)

**⚠️ PREREQUISITE: All data preprocessing cells above must be executed first!**

This section runs SBOA optimization if enabled, otherwise uses default hyperparameters.

## Step 1: Load & Preprocess Data
Run the cells below to load the dataset and prepare features and labels for training.

In [7]:
# Run Hyperparameter Tuning (Conditional)
# This will either run SBOA optimization or use default hyperparameters
# NOTE: This cell requires data to be preprocessed (x_train and y_train must exist)

print('\nInitializing Hyperparameter Tuning Process...')
print('=' * 70)

# Call the wrapper function to get hyperparameters
# Variables x_train and y_train should already exist from preprocessing
try:
    optimized_hyperparams = get_hyperparameters(
        enable_sboa=ENABLE_SBOA,
        x_train=x_train,
        y_train=y_train,
        sboa_config=SBOA_CONFIG,
        verbose=True
    )
    print('\n✓ Hyperparameter selection complete!')
    
except NameError as e:
    print(f'\n❌ Error: {e}')
    print('\n' + '=' * 70)
    print('DATA NOT LOADED - PLEASE RUN PREPROCESSING CELLS FIRST')
    print('=' * 70)
    print('\nRequired cells to run (in order):')
    print('  1. Load raw dataset files')
    print('  2. Define attack categorization function')
    print('  3. Preprocess attack categories and features')
    print('  4. Split data into train, validation, and test sets')
    print('  5. Scale numerical features and encode categorical features')
    print('  6. Encode target labels and compute class weights')
    print('\nAfter running all preprocessing cells above, re-run THIS cell.')
    print('=' * 70)
    
    print('\n\nFalling back to default hyperparameters.')
    optimized_hyperparams = DEFAULT_HYPERPARAMS.copy()

print('\n' + '=' * 70)


Initializing Hyperparameter Tuning Process...

❌ Error: name 'x_train' is not defined

DATA NOT LOADED - PLEASE RUN PREPROCESSING CELLS FIRST

Required cells to run (in order):
  1. Load raw dataset files
  2. Define attack categorization function
  3. Preprocess attack categories and features
  4. Split data into train, validation, and test sets
  5. Scale numerical features and encode categorical features
  6. Encode target labels and compute class weights

After running all preprocessing cells above, re-run THIS cell.


Falling back to default hyperparameters.



In [8]:
# Build Final BiGRU Model with Optimized Hyperparameters
# Uses either SBOA-optimized or default hyperparameters

print('\nBuilding BiGRU Model with Optimized Hyperparameters...')
print('=' * 70)

n_features = x_train.shape[1]
n_classes = y_train.shape[1]

# Use the build_bigru_model function with optimized hyperparameters
model = build_bigru_model(optimized_hyperparams, n_features, n_classes)

print(f'\n✓ Model created successfully!')
print(f'Input features: {n_features}')
print(f'Output classes: {n_classes}')
print(f'\nModel Architecture Summary:')
print(f'  BiGRU Layer 1 Units: {optimized_hyperparams["bigru_units_1"]}')
print(f'  BiGRU Layer 2 Units: {optimized_hyperparams["bigru_units_2"]}')
print(f'  BiGRU Layer 3 Units: {optimized_hyperparams["bigru_units_3"]}')
print(f'  Dense Layer 1 Units: {optimized_hyperparams["dense_units_1"]} (Dropout: {optimized_hyperparams["dropout_rate_1"]})')
print(f'  Dense Layer 2 Units: {optimized_hyperparams["dense_units_2"]} (Dropout: {optimized_hyperparams["dropout_rate_2"]})')
print(f'  Dense Layer 3 Units: {optimized_hyperparams["dense_units_3"]} (Dropout: {optimized_hyperparams["dropout_rate_3"]})')
print(f'  Learning Rate: {optimized_hyperparams["learning_rate"]}')
print(f'  Batch Size: {optimized_hyperparams["batch_size"]}')
print('=' * 70)

model.summary()


Building BiGRU Model with Optimized Hyperparameters...


NameError: name 'x_train' is not defined

In [9]:
# Hyperparameter Tuning Report
# Display comparison between default and SBOA-optimized parameters (if SBOA was enabled)

print('\n' + '=' * 70)
print('HYPERPARAMETER TUNING REPORT')
print('=' * 70)

if ENABLE_SBOA:
    print('\nComparison: Default vs SBOA-Optimized Parameters\n')
    print(f'{"Parameter":<25} {"Default":<20} {"Optimized":<20}')
    print('-' * 65)
    
    for param in DEFAULT_HYPERPARAMS.keys():
        default_val = DEFAULT_HYPERPARAMS[param]
        optimized_val = optimized_hyperparams[param]
        
        # Format for readability
        default_str = f'{default_val:.6f}' if isinstance(default_val, float) else str(default_val)
        optimized_str = f'{optimized_val:.6f}' if isinstance(optimized_val, float) else str(optimized_val)
        
        print(f'{param:<25} {default_str:<20} {optimized_str:<20}')
    
    print('-' * 65)
    print('\n✓ SBOA Optimization was applied!')
    print('Note: Training metrics may improve with optimized hyperparameters.')
else:
    print('\n✓ Using Default Hyperparameters')
    print('To enable SBOA optimization, set ENABLE_SBOA = True')
    print('  (Note: SBOA optimization takes 1-2 hours)')

print('\n' + '=' * 70)


HYPERPARAMETER TUNING REPORT

✓ Using Default Hyperparameters
To enable SBOA optimization, set ENABLE_SBOA = True
  (Note: SBOA optimization takes 1-2 hours)



---

## Section 4: Model Building & Training

Build the BiGRU model with optimized hyperparameters and train it.

In [10]:
# GCN Mapping: Create fully connected edge index for graph neural network
def create_fully_connected_edge_index(num_nodes):
    """
    Creates a fully connected graph structure for GCN.
    Every node is connected to every other node.
    
    Args:
        num_nodes: Number of nodes (batch size or feature dimension)
    
    Returns:
        edge_index: Tensor of shape [2, num_edges] representing graph connectivity
    """
    if num_nodes < 2:
        edge_index = torch.tensor([[0], [0]], dtype=torch.long)
    else:
        edge_list = list(combinations(range(num_nodes), 2))
        edge_index = torch.tensor([
            [src for src, dst in edge_list] + [dst for src, dst in edge_list],
            [dst for src, dst in edge_list] + [src for src, dst in edge_list],
        ], dtype=torch.long)
    return edge_index

## Section 2: Data Loading & Preprocessing

**RUN ALL CELLS IN THIS SECTION BEFORE HYPERPARAMETER TUNING**

These cells load either the UNSW-NB15 or CICIDS dataset, preprocess features, and prepare data for model training.

In [11]:
# Load raw dataset files (UNSW-NB15 or CICIDS)
from pathlib import Path

# Configuration
MAX_ROWS_PER_FILE = 100000  # Per input file
DATASET_OPTION = 'cicids'     # 'unsw' or 'cicids'

unsw_dir = Path('./dataset/unsw')
cicids_dir = Path('./dataset/cicids')
dataset_option = DATASET_OPTION.strip().lower()

def read_csv_with_fallback(path, **kwargs):
    """Read CSV robustly by trying common encodings used in IDS datasets."""
    encodings = ['utf-8', 'ISO-8859-1', 'cp1252']
    last_error = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except UnicodeDecodeError as err:
            last_error = err
    raise last_error

if dataset_option == 'unsw':
    # Load UNSW feature column names
    dataset_columns = pd.read_csv(unsw_dir / 'NUSW-NB15_features.csv', encoding='ISO-8859-1')
    feature_names = dataset_columns['Name'].tolist()

    # Load UNSW split files
    parts = []
    for i in range(1, 5):
        file_path = unsw_dir / f'UNSW-NB15_{i}.csv'
        df = pd.read_csv(
            file_path,
            header=None,
            names=feature_names,
            nrows=MAX_ROWS_PER_FILE,
            low_memory=False
        )
        parts.append(df)

    combined_data = pd.concat(parts, ignore_index=True)
    del parts
    active_dataset = 'UNSW-NB15'

elif dataset_option == 'cicids':
    # Load all CICIDS CSV files from the CICIDS dataset directory
    cicids_files = sorted(cicids_dir.glob('*.csv'))

    if not cicids_files:
        raise FileNotFoundError(
            'No CICIDS CSV files found in ./dataset/cicids.'
        )

    parts = []
    for file_path in cicids_files:
        df = read_csv_with_fallback(file_path, nrows=MAX_ROWS_PER_FILE, low_memory=False)
        parts.append(df)

    combined_data = pd.concat(parts, ignore_index=True)
    del parts
    combined_data.columns = [str(col).strip() for col in combined_data.columns]
    active_dataset = 'CICIDS'

else:
    raise ValueError("DATASET_OPTION must be either 'unsw' or 'cicids'.")

print(f'Dataset option: {DATASET_OPTION}')
print(f'UNSW path: {unsw_dir.resolve()}')
print(f'CICIDS path: {cicids_dir.resolve()}')
print(f'Active dataset: {active_dataset}')
print(f'Total rows loaded: {len(combined_data)}')
print(f'Total columns: {combined_data.shape[1]}')

Dataset option: cicids
UNSW path: C:\Users\ashis\OneDrive\phd\Paper1\code\dataset\unsw
CICIDS path: C:\Users\ashis\OneDrive\phd\Paper1\code\dataset\cicids
Active dataset: CICIDS
Total rows loaded: 800000
Total columns: 85


In [12]:
# Define attack categorization function
def group_attacks_cicids(label):
    """
    Map detailed attack labels to broader categories.
    
    Categories:
    - Benign: Normal traffic
    - Ds/DDos: DDoS and DoS attacks
    - Brute Force: SSH/FTP password attacks
    - Web attack: Web application attacks
    - Malware: All other malicious activity
    """
    label = str(label).strip()
    
    if label == 'BENIGN':
        return 'Benign'
    elif label in ['DDoS', 'DoS Slowhttptest', 'DoS slowloris', 'DoS GoldenEye', 'DoS Hulk']:
        return 'Ds/DDos'
    elif label in ['FTP-Patator', 'SSH-Patator']:
        return 'Brute Force'
    elif label in ['Web Attack Brute Force', 'Web Attack Sql Injection', 'Web Attack XSS']:
        return 'Web attack'
    else:
        return 'Malware'

In [13]:
# Preprocess attack categories and features
combined_data.columns = [str(col).strip() for col in combined_data.columns]

# Ensure a unified target column named attack_cat
if 'attack_cat' not in combined_data.columns:
    label_candidates = ['Label', 'label']
    found_label = next((c for c in label_candidates if c in combined_data.columns), None)
    if found_label is None:
        raise ValueError("Could not find target column. Expected 'attack_cat' (UNSW) or 'Label' (CICIDS).")
    combined_data['attack_cat'] = combined_data[found_label].astype(str)

# Clean and categorize attack labels
combined_data['attack_cat'] = combined_data['attack_cat'].fillna('normal').astype(str).apply(lambda x: x.strip())

# Detect if dataset contains CICIDS attack types
cicids_markers = {'BENIGN', 'DDoS', 'DoS Slowhttptest', 'DoS slowloris', 'DoS GoldenEye', 'DoS Hulk', 
                  'FTP-Patator', 'SSH-Patator', 'Web Attack Brute Force', 'Web Attack Sql Injection', 'Web Attack XSS'}

if set(combined_data['attack_cat'].unique()) & cicids_markers:
    # Apply CICIDS grouping
    combined_data['attack_cat'] = combined_data['attack_cat'].apply(group_attacks_cicids)
else:
    # Apply UNSW/generic grouping
    combined_data['attack_cat'] = combined_data['attack_cat'].str.lower()
    combined_data['attack_cat'] = combined_data['attack_cat'].replace('backdoors', 'backdoor', regex=True).str.lower()

# UNSW-specific cleanup: apply only when columns exist
if 'ct_flw_http_mthd' in combined_data.columns:
    combined_data['ct_flw_http_mthd'] = combined_data['ct_flw_http_mthd'].fillna(0)
if 'is_ftp_login' in combined_data.columns:
    combined_data['is_ftp_login'] = combined_data['is_ftp_login'].fillna(0)
    combined_data['is_ftp_login'] = np.where(combined_data['is_ftp_login'] > 1, 1, combined_data['is_ftp_login'])
if 'service' in combined_data.columns:
    combined_data['service'] = combined_data['service'].astype(str).apply(lambda x: 'None' if x == '-' else x)
if 'ct_ftp_cmd' in combined_data.columns:
    combined_data['ct_ftp_cmd'] = combined_data['ct_ftp_cmd'].replace(to_replace=' ', value=0).astype(float).astype(int)

# Drop non-informative identifier columns when present
drop_if_present = [
    'srcip', 'sport', 'dstip', 'dsport', 'Label',
    'Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Timestamp'
]
cols_to_drop = [c for c in drop_if_present if c in combined_data.columns]
if cols_to_drop:
    combined_data.drop(columns=cols_to_drop, inplace=True)

print(f'Attack categories: {combined_data["attack_cat"].unique()}')
print(f'Rows after preprocessing: {len(combined_data)}')
print(f'Columns after preprocessing: {combined_data.shape[1]}')

Attack categories: ['Benign' 'Ds/DDos' 'Malware' 'Brute Force']
Rows after preprocessing: 800000
Columns after preprocessing: 79


In [14]:
# Split data into train, validation, and test sets
# Stratified split to maintain class distribution

# 80% train+val, 20% test
train, test = train_test_split(
    combined_data,
    test_size=0.2,
    random_state=16,
    stratify=combined_data['attack_cat']
)

# Split train into 80% train, 20% validation
train, val = train_test_split(
    train,
    test_size=0.2,
    random_state=16,
    stratify=train['attack_cat']
)

# Separate features (X) and labels (y)
x_train, y_train = train.drop(columns=['attack_cat']), train[['attack_cat']]
x_test, y_test = test.drop(columns=['attack_cat']), test[['attack_cat']]
x_val, y_val = val.drop(columns=['attack_cat']), val[['attack_cat']]

print(f'Train: {x_train.shape}, Val: {x_val.shape}, Test: {x_test.shape}')

Train: (512000, 78), Val: (128000, 78), Test: (160000, 78)


In [ ]:
# Scale numerical features and encode categorical features
# Check if data is already in numpy array format (from previous preprocessing)
if isinstance(x_train, np.ndarray):
    # Data is already numpy arrays - ensure proper dtype
    x_train = x_train.astype(np.float32)
    x_test = x_test.astype(np.float32)
    x_val = x_val.astype(np.float32)
    
    print(f'Feature shapes - Train: {x_train.shape}, Val: {x_val.shape}, Test: {x_test.shape}')
else:
    # Data is pandas DataFrame - need to handle categorical/numerical columns
    x_train = x_train.copy()
    x_test = x_test.copy()
    x_val = x_val.copy()

    # Detect categorical columns dynamically for cross-dataset compatibility
    cat_col = x_train.select_dtypes(include=['object', 'category']).columns.tolist()
    num_col = [col for col in x_train.columns if col not in cat_col]

    # Convert numeric-like columns and sanitize problematic values
    for col in num_col:
        x_train[col] = pd.to_numeric(x_train[col], errors='coerce')
        x_test[col] = pd.to_numeric(x_test[col], errors='coerce')
        x_val[col] = pd.to_numeric(x_val[col], errors='coerce')

        # Replace inf with NaN before imputation
        x_train[col] = x_train[col].replace([np.inf, -np.inf], np.nan)
        x_test[col] = x_test[col].replace([np.inf, -np.inf], np.nan)
        x_val[col] = x_val[col].replace([np.inf, -np.inf], np.nan)

        # Optional clipping to avoid extreme overflow values
        x_train[col] = x_train[col].clip(lower=-1e12, upper=1e12)
        x_test[col] = x_test[col].clip(lower=-1e12, upper=1e12)
        x_val[col] = x_val[col].clip(lower=-1e12, upper=1e12)

    # Fill missing values after numeric conversion
    for col in num_col:
        median_val = x_train[col].median()
        if pd.isna(median_val):
            median_val = 0.0
        x_train[col] = x_train[col].fillna(median_val)
        x_test[col] = x_test[col].fillna(median_val)
        x_val[col] = x_val[col].fillna(median_val)

    # Standardize numerical features
    if num_col:
        scaler = StandardScaler().fit(x_train[num_col])
        x_train[num_col] = scaler.transform(x_train[num_col])
        x_test[num_col] = scaler.transform(x_test[num_col])
        x_val[num_col] = scaler.transform(x_val[num_col])

    # One-hot encode categorical features when available
    if cat_col:
        ct = ColumnTransformer(
            transformers=[('encoder', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), cat_col)],
            remainder='passthrough'
        )
        x_train = np.array(ct.fit_transform(x_train), dtype=np.float32)
        x_test = np.array(ct.transform(x_test), dtype=np.float32)
        x_val = np.array(ct.transform(x_val), dtype=np.float32)
    else:
        x_train = x_train.to_numpy(dtype=np.float32)
        x_test = x_test.to_numpy(dtype=np.float32)
        x_val = x_val.to_numpy(dtype=np.float32)

    print(f'Categorical columns: {len(cat_col)} | Numerical columns: {len(num_col)}')
    print(f'Encoded feature shapes - Train: {x_train.shape}, Val: {x_val.shape}, Test: {x_test.shape}')

AttributeError: 'numpy.ndarray' object has no attribute 'select_dtypes'

In [16]:
# Encode target labels and compute class weights
# Get unique attack types from training data
attacks = np.sort(y_train['attack_cat'].unique())

# One-hot encode target labels (handles rare labels absent in validation/test)
ct1 = ColumnTransformer(
    transformers=[('encoder', OneHotEncoder(categories=[attacks], sparse_output=False, handle_unknown='ignore'), ['attack_cat'])],
    remainder='passthrough'
)
y_train = np.array(ct1.fit_transform(y_train), dtype=np.float32)
y_test = np.array(ct1.transform(y_test), dtype=np.float32)
y_val = np.array(ct1.transform(y_val), dtype=np.float32)

# Compute balanced class weights to handle imbalanced dataset
y_train_idx = np.argmax(y_train, axis=1)
classes = np.arange(y_train.shape[1])
class_weights_values = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train_idx
)
class_weights = {int(i): float(w) for i, w in enumerate(class_weights_values)}

print(f'Encoded target shapes - Train: {y_train.shape}, Val: {y_val.shape}, Test: {y_test.shape}')
print(f'Attack classes: {attacks}')
print(f'Class weights: {class_weights}')

Encoded target shapes - Train: (512000, 4), Val: (128000, 4), Test: (160000, 4)
Attack classes: ['Benign' 'Brute Force' 'Ds/DDos' 'Malware']
Class weights: {0: 0.29407507644803255, 1: 25.2465483234714, 2: 2.065015729612003, 3: 13.220409006403635}


In [17]:
# Build Bidirectional GRU (BiGRU) model architecture
# Architecture: Input → BiGRU layers → Dense layers → Output
n_features = x_train.shape[1]

model = Sequential([
    Input(shape=(n_features,)),
    Reshape((n_features, 1)),
    Reshape((1, n_features)),
    
    # BiGRU layers: Process sequence forward and backward
    Bidirectional(GRU(32, return_sequences=True)),
    BatchNormalization(),
    
    Bidirectional(GRU(64, return_sequences=True)),
    BatchNormalization(),
    
    Bidirectional(GRU(64)),
    BatchNormalization(),
    
    # Dense layers: Classification head
    Dense(128, activation='relu'),
    Dropout(0.3),
    
    Dense(64, activation='relu'),
    Dropout(0.2),
    
    Dense(32, activation='relu'),
    Dropout(0.2),
    
    Dense(y_train.shape[1], activation='softmax')  # Output layer (multi-class)
])

# Compile with Adam optimizer and categorical cross-entropy loss
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('Model Architecture:')
print(f'Input features: {n_features}')
print(f'Output classes: {y_train.shape[1]}')
model.summary()

Model Architecture:
Input features: 78
Output classes: 4


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ reshape (Reshape)               │ (None, 78, 1)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_1 (Reshape)             │ (None, 1, 78)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 1, 64)          │        21,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1, 64)          │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 1, 128)         │        49,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 1, 128)         │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 128)            │        74,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 174,180 (680.39 KB)

 Trainable params: 173,540 (677.89 KB)

 Non-trainable params: 640 (2.50 KB)

In [19]:
# Configure training callbacks
# Early stopping: Stop if validation accuracy doesn't improve
# Learning rate reduction: Reduce LR if validation loss plateaus
callbacks = [
    EarlyStopping(
        monitor='val_accuracy',
        mode='max',
        patience=8,  # Stop after 8 epochs without improvement
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,  # Reduce LR by 50%
        patience=3,  # Wait 3 epochs before reducing
        min_lr=1e-6,
        verbose=1
    )
]

print('Callbacks configured:')
print('- Early Stopping (patience=8 epochs)')
print('- Learning Rate Reduction (factor=0.5, patience=3 epochs)')

Callbacks configured:
- Early Stopping (patience=8 epochs)
- Learning Rate Reduction (factor=0.5, patience=3 epochs)


In [ ]:
# Train the BiGRU model
print('Starting training...\n')

# Safer defaults for large CICIDS runs to reduce kernel crash risk (OOM)
default_batch = int(optimized_hyperparams.get('batch_size', 256))
if 'active_dataset' in globals() and str(active_dataset).lower() == 'cicids':
    EPOCHS = 30
    BATCH_SIZE = min(default_batch, 64)
else:
    EPOCHS = 60
    BATCH_SIZE = default_batch

# Build tf.data pipelines for more stable memory usage
shuffle_buffer = min(len(x_train), 100000)
train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(shuffle_buffer)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
val_ds = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print(f'Dataset: {active_dataset if "active_dataset" in globals() else "unknown"}')
print(f'Epochs: {EPOCHS} | Batch size: {BATCH_SIZE}')

history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=2
)

print('\n✓ Training completed!')

Starting training...

Dataset: CICIDS
Epochs: 30 | Batch size: 64
Epoch 1/30
8000/8000 - 45s - 6ms/step - accuracy: 0.9170 - loss: 0.1577 - val_accuracy: 0.9333 - val_loss: 0.1887 - learning_rate: 1.0000e-03
Epoch 2/30
8000/8000 - 36s - 4ms/step - accuracy: 0.9424 - loss: 0.1094 - val_accuracy: 0.9047 - val_loss: 0.1858 - learning_rate: 1.0000e-03
Epoch 3/30
8000/8000 - 36s - 4ms/step - accuracy: 0.9539 - loss: 0.0951 - val_accuracy: 0.9436 - val_loss: 0.1495 - learning_rate: 1.0000e-03
Epoch 4/30
8000/8000 - 37s - 5ms/step - accuracy: 0.9550 - loss: 0.0886 - val_accuracy: 0.9625 - val_loss: 0.1201 - learning_rate: 1.0000e-03
Epoch 5/30
8000/8000 - 36s - 5ms/step - accuracy: 0.9555 - loss: 0.0873 - val_accuracy: 0.9704 - val_loss: 0.0753 - learning_rate: 1.0000e-03
Epoch 6/30
8000/8000 - 34s - 4ms/step - accuracy: 0.9593 - loss: 0.0815 - val_accuracy: 0.9753 - val_loss: 0.1175 - learning_rate: 1.0000e-03
Epoch 7/30
8000/8000 - 36s - 4ms/step - accuracy: 0.9544 - loss: 0.0859 - val_accu

In [ ]:
# Evaluate model on test set
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)

print('Model Evaluation on Test Set')
print('=' * 50)
print(f'Test Loss:     {test_loss:.4f}')
print(f'Test Accuracy: {test_acc*100:.2f}%')
print('=' * 50)

In [ ]:
# Visualize training history with 4 metrics: Loss, Accuracy, Precision, Recall
# Create 2x2 subplot grid

from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

plt.figure(figsize=(14, 10))

# 1. Loss (Top Left)
plt.subplot(2, 2, 1)
plt.plot(history.history['loss'], label='Train Loss', linewidth=2, marker='o', markersize=3)
plt.plot(history.history['val_loss'], label='Val Loss', linewidth=2, marker='s', markersize=3)
plt.title('Model Loss Over Epochs', fontsize=13, fontweight='bold')
plt.xlabel('Epoch', fontsize=11)
plt.ylabel('Loss', fontsize=11)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

# 2. Accuracy (Top Right)
plt.subplot(2, 2, 2)
plt.plot(history.history['accuracy'], label='Train Accuracy', linewidth=2, marker='o', markersize=3)
plt.plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2, marker='s', markersize=3)
plt.title('Model Accuracy Over Epochs', fontsize=13, fontweight='bold')
plt.xlabel('Epoch', fontsize=11)
plt.ylabel('Accuracy', fontsize=11)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

# 3. Precision (Bottom Left)
# Calculate precision on validation set (using final model predictions)
y_val_pred_final = model.predict(x_val, verbose=0)
y_val_pred_classes = np.argmax(y_val_pred_final, axis=1)
y_val_true_classes = np.argmax(y_val, axis=1)
precision_val = precision_score(y_val_true_classes, y_val_pred_classes, average='weighted', zero_division=0)

# Calculate precision on test set
y_test_pred = model.predict(x_test, verbose=0)
y_test_pred_classes = np.argmax(y_test_pred, axis=1)
y_test_true_classes = np.argmax(y_test, axis=1)
precision_test = precision_score(y_test_true_classes, y_test_pred_classes, average='weighted', zero_division=0)

plt.subplot(2, 2, 3)
plt.bar(['Validation', 'Test'], [precision_val, precision_test], color=['#FF7F0E', '#1F77B4'], alpha=0.8, edgecolor='black', linewidth=1.5)
plt.title('Model Precision', fontsize=13, fontweight='bold')
plt.ylabel('Precision Score', fontsize=11)
plt.ylim([0, 1.0])
plt.text(0, precision_val + 0.02, f'{precision_val:.4f}', ha='center', fontsize=10, fontweight='bold')
plt.text(1, precision_test + 0.02, f'{precision_test:.4f}', ha='center', fontsize=10, fontweight='bold')
plt.grid(True, alpha=0.3, axis='y')

# 4. Recall (Bottom Right)
recall_val = recall_score(y_val_true_classes, y_val_pred_classes, average='weighted', zero_division=0)
recall_test = recall_score(y_test_true_classes, y_test_pred_classes, average='weighted', zero_division=0)

plt.subplot(2, 2, 4)
plt.bar(['Validation', 'Test'], [recall_val, recall_test], color=['#2CA02C', '#D62728'], alpha=0.8, edgecolor='black', linewidth=1.5)
plt.title('Model Recall', fontsize=13, fontweight='bold')
plt.ylabel('Recall Score', fontsize=11)
plt.ylim([0, 1.0])
plt.text(0, recall_val + 0.02, f'{recall_val:.4f}', ha='center', fontsize=10, fontweight='bold')
plt.text(1, recall_test + 0.02, f'{recall_test:.4f}', ha='center', fontsize=10, fontweight='bold')
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print('\n✓ Training metrics visualization complete!')
print(f'\nFinal Performance Metrics:')
print(f'  Loss - Val: {history.history["val_loss"][-1]:.4f} | Test: {test_loss:.4f}')
print(f'  Accuracy - Val: {history.history["val_accuracy"][-1]:.4f} | Test: {test_acc:.4f}')
print(f'  Precision - Val: {precision_val:.4f} | Test: {precision_test:.4f}')
print(f'  Recall - Val: {recall_val:.4f} | Test: {recall_test:.4f}')